## Bayes Optimal v Prompting for Combination Lock

In [55]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any

In [56]:
styles = ["1", "2", "3", "4", "5", "6"]
results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}.jsonl'
    for style in styles
]
bayes_optimal_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/bayes_optimal.jsonl'

models = [
    # "Gemini Pro 2.5",
    "DeepSeek R1",
    # "Claude Opus 4",
    # "Claude 3.5 Sonnet",
    # "OpenAI o3",
]

model_ids = [
    # "google/gemini-2.5-pro-preview",
    "deepseek/deepseek-r1-0528",
    # "anthropic/claude-opus-4",
    # "anthropic/claude-3.5-sonnet",
    # "openai/o3",
]

In [57]:
results = []

for style, results_path in zip(styles, results_paths):
    with open(results_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
            model = data['model']
            results.append({
                'game_id': data['game_id'],
                'model': model + f' (s={style})',
                'regret': regret,
                'length': len(data['history']),
                'style': style
            })
results_df = pd.DataFrame(results)

bayes_optimal = []

with open(bayes_optimal_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
        bayes_optimal.append({
            'game_id': data['game_id'],
            'model': 'Bayes Optimal',
            'regret': regret,
            'length': len(data['history']),
            'style': style
        })
bayes_optimal_df = pd.DataFrame(bayes_optimal)

In [58]:
results_df.head(3)

,game_id,model,regret,length,style
0,21,anthropic/claude-3.5-sonnet (s=1),"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,1
1,22,anthropic/claude-3.5-sonnet (s=1),"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",4,1
2,67,anthropic/claude-3.5-sonnet (s=1),"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]",5,1


In [59]:
bayes_optimal_df.head(3)

,game_id,model,regret,length,style
0,0,Bayes Optimal,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]",10,6
1,1,Bayes Optimal,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]",10,6
2,2,Bayes Optimal,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]",6,6


In [60]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
bayes_optimal_df = bayes_optimal_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [61]:
len(results_df), len(bayes_optimal_df)

(1400, 100)

In [62]:
cumulative_regrets = []
error_bars = []

for style in styles:
    for mi, model in enumerate(models):
        model_df = results_df[results_df['model'] == model_ids[mi] + f' (s={style})']
        # Convert regret lists to numpy array for easier computation
        regret_array = np.array(model_df['regret'].values.tolist())
        
        # Calculate mean regret per turn
        model_regret = np.mean(regret_array, axis=0)
        cumulative_regret = np.cumsum(model_regret)
        cumulative_regrets.append(cumulative_regret)
        
        # Calculate standard error of the mean for each turn
        sem = np.std(regret_array, axis=0) / np.sqrt(len(model_df))
        cumulative_sem = np.cumsum(sem)
        error_bars.append(cumulative_sem)
        
        print(f'{model} cumulative regret: {cumulative_regret}')

bayes_optimal_regret = np.mean(np.array(bayes_optimal_df['regret'].values.tolist()), axis=0)
bayes_optimal_cumulative_regret = np.cumsum(bayes_optimal_regret)

DeepSeek R1 cumulative regret: [1.   1.99 2.93 3.82 4.68 5.44 6.1  6.59 6.99 7.33 7.61 7.85]
DeepSeek R1 cumulative regret: [1.   2.   2.97 3.91 4.75 5.51 6.15 6.65 7.11 7.49 7.77 7.97]
DeepSeek R1 cumulative regret: [1.   1.99 2.95 3.84 4.58 5.14 5.5  5.68 5.75 5.8  5.81 5.82]
DeepSeek R1 cumulative regret: [1.   1.98 2.94 3.81 4.58 5.14 5.53 5.78 5.91 5.97 5.99 6.  ]
DeepSeek R1 cumulative regret: [1.   2.   2.98 3.9  4.66 5.22 5.55 5.74 5.81 5.85 5.86 5.86]
DeepSeek R1 cumulative regret: [1.   2.   2.95 3.83 4.53 5.03 5.39 5.63 5.72 5.78 5.81 5.82]


In [63]:
cumulative_regrets

[array([1.  , 1.99, 2.93, 3.82, 4.68, 5.44, 6.1 , 6.59, 6.99, 7.33, 7.61,
        7.85]),
 array([1.  , 2.  , 2.97, 3.91, 4.75, 5.51, 6.15, 6.65, 7.11, 7.49, 7.77,
        7.97]),
 array([1.  , 1.99, 2.95, 3.84, 4.58, 5.14, 5.5 , 5.68, 5.75, 5.8 , 5.81,
        5.82]),
 array([1.  , 1.98, 2.94, 3.81, 4.58, 5.14, 5.53, 5.78, 5.91, 5.97, 5.99,
        6.  ]),
 array([1.  , 2.  , 2.98, 3.9 , 4.66, 5.22, 5.55, 5.74, 5.81, 5.85, 5.86,
        5.86]),
 array([1.  , 2.  , 2.95, 3.83, 4.53, 5.03, 5.39, 5.63, 5.72, 5.78, 5.81,
        5.82])]

In [64]:
# Use Plotly's Dark24 color set for darker colors
colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]

# Set LaTeX font for all text elements
latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

fig = go.Figure()

for si, style in enumerate(styles):
    for mi, model in enumerate(models):
        x_vals = list(range(1, 13))
        y_mean = cumulative_regrets[mi + si]
        y_err = error_bars[mi + si]
        color = colors[(mi + si) % len(colors)]

        # Add shaded error region (as a filled area)
        fig.add_trace(go.Scatter(
            x=x_vals + x_vals[::-1],
            y=(y_mean + y_err).tolist() + (y_mean - y_err)[::-1].tolist(),
            fill='toself',
            fillcolor=f'rgba{tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + (0.18,)}',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False,
            name=f"{model} (s={style})"
        ))

        # Add main mean curve, thicker
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_mean,
            mode='lines+markers',
            name=model + f' (s={style})',
            line=dict(width=4, color=color),
            marker=dict(size=6, color=color)
        ))

# Add Bayes Optimal line
fig.add_trace(go.Scatter(
    x=list(range(1, 13)),
    y=bayes_optimal_cumulative_regret,
    mode='markers+lines+lines',
    name='Bayes Optimal',
    line=dict(width=4, color='rgba(0, 0, 0, 0.8)'),
    marker=dict(size=6, color='rgba(0, 0, 0, 0.8)')
))

# Add y=x baseline as a dashed line
baseline_x = list(range(1, 13))
baseline_y = list(range(1, 13))
fig.add_trace(go.Scatter(
    x=baseline_x,
    y=baseline_y,
    mode='lines',
    name='Baseline',
    line=dict(color='black', width=2, dash='dash'),
    showlegend=True
))

fig.update_layout(
    width=600,
    height=470,
    title=dict(
        text='',
        font=latex_font
    ),
    xaxis_title="Episode (Symbol Combo-Lock)",
    yaxis_title="Cumulative Regret",
    font=latex_font,
    xaxis=dict(
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis=dict(
        range=[1, 12],
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        side='right',  # default, but we want ticks on both sides
        showticksuffix='all',
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis2=dict(
        overlaying='y',
        side='right',
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    legend=dict(
        title='',
        x=0.03,  # left edge, inside plot
        y=0.97,  # top edge, inside plot
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        font=latex_font
    ),
    template='plotly_white'
)

# Add yaxis2 to all traces so ticks show on both sides
for trace in fig.data:
    trace.update(yaxis='y')

In [65]:
fig.show()

In [50]:
# TODOs

# Bayes-optimal baseline [DONE]
# Error bars [DONE]
# Efficiency (tokens per game for each model per episode)